# Stage 00b — Figure Matching

Assigns figure labels to every downloaded image using **pure positional matching**
(no OCR, no external models).  The Nth crop is matched to the Nth key in the
Brief Description of the Drawings from the PatSeer Excel.

**Output naming convention:**

| Pattern | Meaning |
|---------|----------|
| `US…_img003_F002B.png` | img file, matched to FIG. 2B |
| `US…_img003_Fu001.png` | img file, count mismatch — needs review |
| `US…_D00005_crop01_F001.png` | D file crop, matched |
| `US…_FAT001_crop01_Fu003.png` | FAT file crop, unmatched |

D and FAT originals are **always kept** alongside their crops.

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and Excel index |
| 2 | Dry-run on one patent (no renaming) |
| 3 | Full run — match and rename all patents |
| 4 | Print matching report |

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config_loader import load_config
from src.figure_matcher import (
    parse_description_figures,
    detect_split_needed,
    match_positionally,
    rename_matched_files,
)
from src.extractor import load_patseer_excel

cfg         = load_config()
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
print(f"Loaded {len(excel_index)} patents from Excel.")

In [ ]:
# ─── Dry-run on ONE patent — inspect results before committing any renames ────
TEST_PATENT = "US2015014475A1"   # change to any patent ID present in raw/

raw_dir         = cfg["paths"]["raw_images"]
img_dir         = raw_dir / TEST_PATENT
meta            = excel_index.get(TEST_PATENT, {})
description_text = meta.get("description_of_drawings") or ""

print(f"Patent      : {TEST_PATENT}")
print(f"Img dir     : {img_dir}")
print(f"Dir exists  : {img_dir.exists()}")
print(f"Desc length : {len(description_text)} chars")
print()

# Parse description
fig_keys = parse_description_figures(description_text)
print(f"Description figures ({len(fig_keys)}): {fig_keys}")
print()

# File inventory
import re
img_files  = sorted(img_dir.glob(f"{TEST_PATENT}_img*.png"))
d_files    = sorted(img_dir.glob(f"{TEST_PATENT}_D*.png"))
fat_files  = sorted(img_dir.glob(f"{TEST_PATENT}_FAT*.png"))
print(f"img files   : {len(img_files)}")
print(f"D files     : {len(d_files)}")
print(f"FAT files   : {len(fat_files)}")
print()

# Split detection for D and FAT
for f in d_files + fat_files:
    n = detect_split_needed(f)
    print(f"  {f.name}  →  {n} sub-figure(s)")
if d_files or fat_files:
    print()

# Run matching (does NOT rename anything)
matches = match_positionally(TEST_PATENT, img_dir, description_text)

total_crops = len(matches)
total_figs  = len(fig_keys)
count_ok    = total_crops == total_figs

print(f"Global crops  : {total_crops}")
print(f"Desc figures  : {total_figs}")
print(f"Count match   : {'✓ EXACT — positional match' if count_ok else '✗ MISMATCH — all will be _Fu'}")
print()

# Preview rename table
import pandas as pd
from IPython.display import display

df_preview = pd.DataFrame([
    {
        "source_file":     m["source_file"],
        "type":            m["source_type"],
        "split":           m["was_split"],
        "fig_key":         m["fig_key"],
        "needs_review":    m["needs_review"],
        "output_filename": m["output_filename"],
    }
    for m in matches
])
display(df_preview)

In [ ]:
# ─── Full run — match and rename all patents in raw/ ─────────────────────────
# Patents without a download folder are silently skipped.

raw_dir     = cfg["paths"]["raw_images"]
all_summary = []
run_errors  = []

for patent_id, meta in excel_index.items():
    img_dir = raw_dir / patent_id
    if not img_dir.exists():
        continue

    description_text = meta.get("description_of_drawings") or ""

    try:
        matches = match_positionally(patent_id, img_dir, description_text)
        if not matches:
            continue

        summary              = rename_matched_files(matches, img_dir)
        summary["patent_id"] = patent_id
        all_summary.append(summary)

        n_F  = summary["renamed_F"]
        n_Fu = summary["renamed_Fu"]
        icon = "✓" if n_Fu == 0 else "⚠"
        print(f"  {icon}  {patent_id:<30}  F={n_F:>4}  Fu={n_Fu:>3}")

    except Exception as exc:
        run_errors.append((patent_id, str(exc)))
        print(f"  ✗  {patent_id}: {exc}")

print(f"\nDone. Patents processed: {len(all_summary)}  Errors: {len(run_errors)}")

In [ ]:
# ─── Matching report ─────────────────────────────────────────────────────────
if not all_summary:
    print("No results yet — run Cell 3 first.")
else:
    import pandas as pd

    df = pd.DataFrame(all_summary)

    total_patents   = len(df)
    perfect         = int((df["renamed_Fu"] == 0).sum())
    needs_review    = int((df["renamed_Fu"] > 0).sum())
    total_F         = int(df["renamed_F"].sum())
    total_Fu        = int(df["renamed_Fu"].sum())
    total_figs      = total_F + total_Fu
    kept_originals  = int(df["kept_originals"].sum())

    def _pct(n):
        return f"{n / total_patents * 100:.0f}%" if total_patents else "N/A"

    print("══════════════════════════════════════")
    print("Figure Matching Report")
    print("══════════════════════════════════════")
    print(f"Patents processed       : {total_patents}")
    print(f"Perfect matches (_F)    : {perfect}  ({_pct(perfect)})")
    print(f"Needs review (_Fu)      : {needs_review}  ({_pct(needs_review)})")
    print(f"Total figures matched   : {total_figs}")
    print(f"  - labeled   (_F)      : {total_F}")
    print(f"  - unlabeled (_Fu)     : {total_Fu}")
    print(f"D/FAT originals kept    : {kept_originals}")
    print(f"Run errors              : {len(run_errors)}")
    print("══════════════════════════════════════")

    if run_errors:
        print("\nErrors:")
        for pid, err in run_errors:
            print(f"  {pid}: {err}")